In [1]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RouterRunnable,RunnableLambda


In [2]:
load_dotenv()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0.9
)


In [3]:
# 3. Create the prompt template
prompt = ChatPromptTemplate.from_template(
    "Suggest ONE creative company name for a business that makes {product}. "
)


In [4]:
# 4. Create the output parser
output_parser = StrOutputParser()


In [5]:
# 5. Build the chain using LCEL
chain = prompt | llm | output_parser


In [6]:
# 6. Run the chain
product = "Queen Size Sheet Set"
result = chain.invoke({"product": product})

print(result)

"RegalRest Linens"


In [7]:
# Chain 1: product -> company name
first_prompt = ChatPromptTemplate.from_template(
    "Suggest ONE creative company name for a business that makes {product}. "
)

In [8]:
chain_one = first_prompt | llm | StrOutputParser()

In [9]:
second_prompt = ChatPromptTemplate.from_template(
    "Write a 20-word description for the following company: {company_name}. "
)
chain_two = second_prompt | llm | StrOutputParser()

In [10]:
# Overall chain: product -> company name -> company description
overall_simple_chain = (
    chain_one
    | (lambda company_name: {"company_name": company_name})
    | chain_two
)


In [11]:
product = "Queen Size Sheet Set"

result = overall_simple_chain.invoke({
    "product": product
})
print(result)

Majestic Dreams Linens offers luxurious, high-quality bedding and linens, transforming your bedroom into a serene sanctuary for restful sleep.


In [12]:
review = "J'adore ce produit. Il est très confortable et la qualité est excellente."

review

"J'adore ce produit. Il est très confortable et la qualité est excellente."

In [13]:
translate_prompt = ChatPromptTemplate.from_template(
    "Translate the following customer review to English. "
    "Return only the translated review, with no explanation.\n\n"
    "Review: {Review}"
)

translate_chain = translate_prompt | llm | output_parser

In [14]:
summary_prompt = ChatPromptTemplate.from_template(
    "Summarize the following customer review in one sentence. "
    "Return only the summary.\n\n"
    "English Review: {English_Review}"
)

summary_chain = summary_prompt | llm | output_parser

In [15]:
language_prompt = ChatPromptTemplate.from_template(
    "What language is the following customer review written in? "
    "Return only the language name.\n\n"
    "Review: {Review}"
)

language_chain = language_prompt | llm | output_parser

In [16]:
followup_prompt = ChatPromptTemplate.from_template(
    "Write a polite follow-up response to the customer based on this review summary. "
    "Write the response in the specified language.\n\n"
    "Summary: {summary}\n\n"
    "Language: {language}\n\n"
    "Return only the follow-up response."
)

followup_chain = followup_prompt | llm | output_parser

In [17]:
overall_chain = (
    RunnablePassthrough.assign(
        English_Review=translate_chain
    )
    .assign(
        summary=summary_chain,
        language=language_chain
    )
    .assign(
        followup_message=followup_chain
    )
)

In [18]:
result = overall_chain.invoke({
    "Review": review
})

result

{'Review': "J'adore ce produit. Il est très confortable et la qualité est excellente.",
 'English_Review': 'I love this product. It is very comfortable and the quality is excellent.',
 'summary': 'The reviewer praises the product for its comfort and excellent quality.',
 'language': 'French',
 'followup_message': "Bonjour,\n\nMerci beaucoup pour votre retour positif ! Nous sommes ravis d'apprendre que vous trouvez notre produit à la fois confortable et de haute qualité. Votre satisfaction est notre priorité, et vos commentaires nous encouragent à continuer d'améliorer nos offres.\n\nSi vous avez d'autres remarques ou suggestions, n'hésitez pas à nous en faire part. Nous serions ravis de vous aider davantage.\n\nCordialement,  \n[Votre Nom]  \n[Votre Poste]  \n[Nom de l'Entreprise]"}

In [19]:
print("Original Review:")
print(result["Review"])

print("\nEnglish Review:")
print(result["English_Review"])

print("\nSummary:")
print(result["summary"])

print("\nLanguage:")
print(result["language"])

print("\nFollow-up Message:")
print(result["followup_message"])

Original Review:
J'adore ce produit. Il est très confortable et la qualité est excellente.

English Review:
I love this product. It is very comfortable and the quality is excellent.

Summary:
The reviewer praises the product for its comfort and excellent quality.

Language:
French

Follow-up Message:
Bonjour,

Merci beaucoup pour votre retour positif ! Nous sommes ravis d'apprendre que vous trouvez notre produit à la fois confortable et de haute qualité. Votre satisfaction est notre priorité, et vos commentaires nous encouragent à continuer d'améliorer nos offres.

Si vous avez d'autres remarques ou suggestions, n'hésitez pas à nous en faire part. Nous serions ravis de vous aider davantage.

Cordialement,  
[Votre Nom]  
[Votre Poste]  
[Nom de l'Entreprise]


In [20]:
type(result)

dict

In [21]:
#Route Chain

In [22]:
physics_template = """
You are a very smart physics professor.
You are great at answering questions about physics in a concise and easy to understand manner.
When you don't know the answer to a question, you admit that you don't know.

Here is a question:
{input}
"""

math_template = """
You are a very good mathematician.
You are great at answering math questions.
You are good because you break down hard problems into component parts,
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{input}
"""

history_template = """
You are a very good historian.
You have excellent knowledge of people, events, and contexts from a range of historical periods.
You use historical evidence to support your explanations and judgements.

Here is a question:
{input}
"""

computer_science_template = """
You are a successful computer scientist.
You are great at answering coding and computer science questions.
You solve problems by describing solutions in clear imperative steps,
and you balance time complexity and space complexity.

Here is a question:
{input}
"""

default_template = """
Answer the following question as helpfully as possible:

{input}
"""

In [23]:
physics_chain = (
    ChatPromptTemplate.from_template(physics_template)
    | llm
    | output_parser
)

math_chain = (
    ChatPromptTemplate.from_template(math_template)
    | llm
    | output_parser
)

history_chain = (
    ChatPromptTemplate.from_template(history_template)
    | llm
    | output_parser
)

computer_science_chain = (
    ChatPromptTemplate.from_template(computer_science_template)
    | llm
    | output_parser
)

default_chain = (
    ChatPromptTemplate.from_template(default_template)
    | llm
    | output_parser
)

In [24]:
router_prompt = ChatPromptTemplate.from_template("""
You are a router. Your job is to classify the user's question into exactly one of the following categories:

- physics: questions about physics, matter, energy, radiation, force, motion, thermodynamics, quantum mechanics
- math: questions about arithmetic, algebra, calculus, geometry, statistics, or mathematical problem solving
- history: questions about historical people, events, periods, civilizations, wars, or historical evidence
- computer science: questions about programming, algorithms, data structures, software engineering, or computer systems
- DEFAULT: anything else

Return only one category name from this list:
physics
math
history
computer science
DEFAULT

Question:
{input}
""")

router_chain = router_prompt | llm | output_parser

In [25]:
def route_input(inputs):
    question = inputs["input"]

    destination = router_chain.invoke({
        "input": question
    }).strip()

    print(f"Router selected: {destination}")

    if destination not in [
        "physics",
        "math",
        "history",
        "computer science",
        "DEFAULT"
    ]:
        destination = "DEFAULT"

    return {
        "key": destination,
        "input": {
            "input": question
        }
    }

In [26]:
router = RouterRunnable({
    "physics": physics_chain,
    "math": math_chain,
    "history": history_chain,
    "computer science": computer_science_chain,
    "DEFAULT": default_chain
})

In [27]:
overall_chain = RunnableLambda(route_input) | router

In [28]:
result = overall_chain.invoke({
    "input": "What is black body radiation?"
})

print(result)

Router selected: physics
Black body radiation refers to the electromagnetic radiation emitted by an idealized physical body (a "black body") that absorbs all incident radiation, regardless of frequency or angle of incidence. A perfect black body is a theoretical concept that serves as a model for understanding thermal radiation.

When a black body is at thermal equilibrium, it emits radiation with a spectrum that depends solely on its temperature. This emission can be described by Planck's law, which shows that the intensity of radiation emitted at different wavelengths increases with temperature. Importantly, the peak wavelength of this radiation shifts according to Wien's displacement law, indicating that hotter bodies emit radiation at shorter wavelengths.

Black body radiation is a fundamental concept in physics, laying the groundwork for quantum mechanics and helping explain phenomena like the ultraviolet catastrophe, which highlighted the limitations of classical physics in expla

In [29]:
result = overall_chain.invoke({
    "input": "What is 2 + 2?"
})

print(result)

Router selected: math
Thank you for the compliment! Let’s break down the question of what \( 2 + 2 \) is.

1. **Identify the numbers involved**: We have two numbers, which are 2 and 2.
2. **Understand the operation**: The operation we are performing is addition.
3. **Perform the addition**: When you add the two numbers together:
   \[
   2 + 2 = 4
   \]

Putting it all together, the answer to \( 2 + 2 \) is \( 4 \).


In [30]:
result = overall_chain.invoke({
    "input": "Why does every cell in our body contain DNA?"
})

print(result)

Router selected: DEFAULT
Every cell in our body contains DNA because it serves as the fundamental blueprint for the organism's development, functioning, and reproduction. Here are several key reasons why DNA is crucial in every cell:

1. **Genetic Information**: DNA carries genetic information that codes for proteins, which are essential for the structure and function of cells. Each gene within the DNA contains instructions for making specific proteins that perform critical roles in the body.

2. **Cellular Identity**: Each cell in the body has a unique role and function, but they all arise from a single fertilized egg, or zygote. The DNA in every cell contains the same genetic code, which ensures that all cells share a common identity while allowing them to differentiate into various cell types (like muscle cells, nerve cells, etc.) based on gene expression.

3. **Replication and Repair**: DNA is capable of replicating itself, ensuring that when cells divide, each new cell receives an